<a href="https://colab.research.google.com/github/SEC-API-io/sec-api-cookbook/blob/main/notebooks/form-nport/swap-derivative-example.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Extract Counterparties of Swap Derivatives from N-PORT Filings

In this tutorial, we will be using Python to extract the counterparties of swap derivatives that are disclosed in SEC Form N-PORT filings. Specifically, we will utilize the `FormNportApi` module from the `sec-api` Python package to access and filter all N-PORT filings published on SEC EDGAR. The `get_data(search_query)` function enables us to search the N-PORT database using any N-PORT form field.

It is worth noting that all N-PORT filings are already converted into JSON, so there is no need for XML to JSON conversion. Before we delve into the code, we will first examine an N-PORT XML example to identify where the names and LEIs of all counterparties can be found.

## N-PORT Example

We will be using the following example filing to illustrate our process:

```
https://www.sec.gov/Archives/edgar/data/1742912/000114554923024925/0001145549-23-024925-index.htm
```

![n-port-filing-details-page-on-sec-edgar](https://i.imgur.com/0dNiaLu.png)

This is the filing details page on SEC EDGAR, which displays three files: HTML, XML, and TXT. Our system has already converted the XML file to JSON and indexed the data in our database, which can be accessed and queried using the `FormNportApi`. We use the Lucene query syntax as the search language.

The process involves two steps. First, we search for all N-PORT filings that include swap derivatives using the following search query:

```
invstOrSecs.derivativeInfo.swapDeriv.derivCat:SWP
```

The `invstOrSecs` array of an N-PORT filing includes objects representing all the securities disclosed by the filer. It contains the information disclosed in "Part C - Schedule of Portfolio Investments" of Form N-PORT. Each object has multiple fields, including "Item C.11 - Derivatives" represented by `derivativeInfo`. This field also represents an object, which varies based on the derivative type. For swaps, the `swapDeriv` field represents an object that contains various fields, including `counterparties` (the information we are interested in), `settlementDt` (the settlement date), and `derivCat` (the derivative category).

The search expression matches and returns all N-PORT filings that have `SWP` as the value in the `invstOrSecs.derivativeInfo.swapDeriv.derivCat` field.

Below is an illustration of a snippet from the original XML file and the corresponding converted JSON data.

| XML Example | XML Converted To JSON |
|-- | -- |
| ![xml-example](https://i.imgur.com/jWvfk6y.png)| ![xml-converted-to-json](https://i.imgur.com/BQOxgCp.png) |

[The documentation outlining all N-PORT form fields is available here.](https://sec-api.io/docs/n-port-data-api#response-structure)

## Extract Counterparties from a Single N-PORT Filing

Before we start, replace `YOUR_API_KEY` in the next cell with the API key shown in your account profile on https://sec-api.io.

In [ ]:
API_KEY = "YOUR_API_KEY"

In [ ]:
!pip install -q sec-api

In [ ]:
from sec_api import FormNportApi

nportApi = FormNportApi(API_KEY)

In [ ]:
search_query = {
  "query": "invstOrSecs.derivativeInfo.swapDeriv.derivCat:SWP AND filedAt:[2023-01-01 TO 2023-04-01]",
  # offset or starting point for search results
  "from": "0",
  # number of filings returned per search request
  "size": "10",
  # sort result by the filedAt field, start with the most recently filed filing
  "sort": [{ "filedAt": { "order": "desc" } }]
}

response = nportApi.get_data(search_query)

The `response` object comprises two fields: `total` and `filings`. The `total` field holds the total number of filings that match our search criteria, while `filings` represents an array of filings themselves. Each N-PORT filing is represented as an object that includes all XML tags and attributes converted to JSON equivalents.

Before we continue, we define a small helper function to print a more readable version of JSON data.

In [ ]:
# Prettyprint JSON
def pprint(item):
  import json
  print(json.dumps(item, indent=2))

Let's access all counterparties mentioned in the swap sections of a single filing.

In [ ]:
swaps = []

sample_filing = response["filings"][1]

for investment in sample_filing["invstOrSecs"]:
  if "derivativeInfo" in investment \
    and "swapDeriv" in investment["derivativeInfo"] \
    and investment["derivativeInfo"]["swapDeriv"]["derivCat"] == "SWP":
    swaps.append(investment["derivativeInfo"]["swapDeriv"])

print("First 3 swap derivatives and their counterparties:")
print("--------------------------------------------------")
pprint(swaps[:3])

First 3 swap derivatives and their counterparties:
--------------------------------------------------
[
  {
    "counterparties": [
      {
        "counterpartyName": "Goldman Sachs International",
        "counterpartyLei": "W22LROWP2IHZNBB6K528"
      }
    ],
    "descRefInstrmnt": {
      "otherRefInst": {
        "issuerName": "N/A",
        "issueTitle": "N/A"
      }
    },
    "swapFlag": "Y",
    "fixedRecDesc": {
      "amount": 0,
      "curCd": "MYR",
      "fixedOrFloating": "Fixed",
      "fixedRt": 3.73
    },
    "floatingPmntDesc": {
      "rtResetTenors": {
        "rtResetTenor": [
          {
            "rateTenor": "Month",
            "rateTenorUnit": "3",
            "resetDt": "Month",
            "resetDtUnit": "3"
          }
        ]
      },
      "curCd": "MYR",
      "fixedOrFloating": "Floating",
      "floatingRtIndex": "Kuala Lumpur Interbank Offer Rate 3 Months",
      "floatingRtSpread": 0,
      "pmntAmt": 0
    },
    "terminationDt": "2028-03-15

Let's import the JSON data into a pandas DataFrame.

In [ ]:
# display 40 columns per DataFrame output in notebook
from google.colab.data_table import DataTable
DataTable.max_columns = 40

In [ ]:
import pandas as pd

swaps_df = pd.json_normalize(swaps)
swaps_df.head(5)

,counterparties,swapFlag,terminationDt,upfrontPmnt,pmntCurCd,upfrontRcpt,rcptCurCd,notionalAmt,curCd,unrealizedAppr,...,fixedPmntDesc.fixedRt,descRefInstrmnt.indexBasketInfo.indexName,descRefInstrmnt.indexBasketInfo.indexIdentifier,otherPmntDesc.value,otherPmntDesc.fixedOrFloating,amtCurSold,curSold,amtCurPur,curPur,settlementDt
0,"[{'counterpartyName': 'Goldman Sachs International', 'counterpartyLei': 'W22LROWP2IHZNBB6K528'}]",Y,2028-03-15,0.0,MYR,0.0,MYR,1.670000e+07,MYR,33579.70,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,"[{'counterpartyName': 'Citibank N.A.', 'counterpartyLei': 'E57ODZWZ7FF32TWEFA76'}]",Y,2023-03-27,0.0,THB,0.0,THB,3.300000e+08,THB,18036.54,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,"[{'counterpartyName': 'LCH Limited', 'counterpartyLei': 'F226TOH6YD6XJB17KS62'}]",Y,2027-08-19,0.0,THB,0.0,THB,9.300000e+07,THB,-20048.39,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,"[{'counterpartyName': 'Bank of America N.A.', 'counterpartyLei': 'B4TYDEB6GKMZO031MB27'}]",Y,2023-02-27,0.0,COP,0.0,COP,6.700000e+10,COP,187180.18,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,"[{'counterpartyName': 'Chicago Mercantile Exchange', 'counterpartyLei': 'SNZ2OJLFK8MNNCLQOF39'}]",Y,2025-11-26,0.0,COP,0.0,COP,1.207920e+10,COP,383305.60,...,4.55,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


The following code generates a list of names and LEIs of all counterparties and removes any duplicate entries.



In [ ]:
counterparty_list = []

# loop through each row in the dataframe and extract the name and lei values
for i, row in swaps_df.iterrows():
    for cp in row['counterparties']:
        counterparty_dict = {'name': cp['counterpartyName'], 'lei': cp['counterpartyLei']}
        counterparty_list.append(counterparty_dict)

# create a new dataframe from the list of dictionaries
counterparties = pd.DataFrame(counterparty_list, columns=['name', 'lei'])

# drop duplicate rows based on the combination of 'name' and 'lei' columns
counterparties = counterparties.drop_duplicates(subset=['name', 'lei'])

print('Swap counterparties of {}'.format(sample_filing['genInfo']['regName']))
counterparties

Swap counterparties of Emerging Markets Local Income Portfolio


,name,lei
0,Goldman Sachs International,W22LROWP2IHZNBB6K528
1,Citibank N.A.,E57ODZWZ7FF32TWEFA76
2,LCH Limited,F226TOH6YD6XJB17KS62
3,Bank of America N.A.,B4TYDEB6GKMZO031MB27
4,Chicago Mercantile Exchange,SNZ2OJLFK8MNNCLQOF39
5,"Intercontinental Exchange, Inc.",5493000F4ZO33MV32P92
26,Goldman Sachs & Co. LLC,FOR8UP27PHTHYVLBNG30
59,Deutsche Bank AG,7LTWFZYICNSX8D621K86
113,JPMorgan Chase Bank N.A.,7H6GLXDRUGQFU57RNE97
138,"Citigroup Global Markets, Inc.",MBNUM2BPBDO7JBLYG310


---

## Extracting Counterparties from All N-PORT Filings of 2020

Our next step is to compile a list of all counterparties mentioned in the swap sections of N-PORT filings filed in 2020. We chose this year because it contains 387 filings with swap derivatives, resulting in a much faster downloading process compared to later years where the number of filings exceeds 6,000 per year. Based on an average API response size of 2.5 MB for 10 filings, we estimate that we will need just under 1 GB of JSON data to analyze the 387 filings from 2020. In contrast, 6,000 filings would result in approximately 15 GB of data.

| Year | # of Filings with Swaps
| -- | -- |
| 2020 | 387 |
| 2021 | 6199 |
| 2022 | 6301 |

The updated Lucene search query looks like this:

```
invstOrSecs.derivativeInfo.swapDeriv.derivCat:SWP AND filedAt:[2020-01-01 TO 2020-12-31]
```


In [ ]:
def extract_counterparties(filing):
  swaps_list = []

  for investment in filing["invstOrSecs"]:
    if "derivativeInfo" in investment \
      and "swapDeriv" in investment["derivativeInfo"] \
      and investment["derivativeInfo"]["swapDeriv"]["derivCat"] == "SWP":
      swaps_list.append(investment["derivativeInfo"]["swapDeriv"])

  swaps_df = pd.json_normalize(swaps_list)

  counterparty_list = []

  # loop through each row in the dataframe and extract the name and lei values
  for i, row in swaps_df.iterrows():
      for cp in row['counterparties']:
          counterparty_dict = {
              'fundName': filing['genInfo']['regName'],
              'fundLei': filing['genInfo']['regLei'],
              'fundCik': filing['genInfo']['regCik'],
              'counterparytName': cp['counterpartyName'],
              'counterpartyLei': cp['counterpartyLei']
              }
          counterparty_list.append(counterparty_dict)

  # create a new dataframe from the list of dictionaries
  counterparties = pd.DataFrame(counterparty_list, columns=['fundName',
                                                            'fundLei',
                                                            'fundCik',
                                                            'counterparytName',
                                                            'counterpartyLei'])

  # drop duplicate rows based on the combination of 'name' and 'lei' columns
  counterparties = counterparties.drop_duplicates(subset=['counterparytName', 'counterpartyLei'])

  counterparties.reset_index(drop=True, inplace=True)

  return counterparties

In [ ]:
result = extract_counterparties(sample_filing)
result

,fundName,fundLei,fundCik,counterparytName,counterpartyLei
0,Emerging Markets Local Income Portfolio,TJHVIYYZOASBZ463H550,0001394395,Goldman Sachs International,W22LROWP2IHZNBB6K528
1,Emerging Markets Local Income Portfolio,TJHVIYYZOASBZ463H550,0001394395,Citibank N.A.,E57ODZWZ7FF32TWEFA76
2,Emerging Markets Local Income Portfolio,TJHVIYYZOASBZ463H550,0001394395,LCH Limited,F226TOH6YD6XJB17KS62
3,Emerging Markets Local Income Portfolio,TJHVIYYZOASBZ463H550,0001394395,Bank of America N.A.,B4TYDEB6GKMZO031MB27
4,Emerging Markets Local Income Portfolio,TJHVIYYZOASBZ463H550,0001394395,Chicago Mercantile Exchange,SNZ2OJLFK8MNNCLQOF39
5,Emerging Markets Local Income Portfolio,TJHVIYYZOASBZ463H550,0001394395,"Intercontinental Exchange, Inc.",5493000F4ZO33MV32P92
6,Emerging Markets Local Income Portfolio,TJHVIYYZOASBZ463H550,0001394395,Goldman Sachs & Co. LLC,FOR8UP27PHTHYVLBNG30
7,Emerging Markets Local Income Portfolio,TJHVIYYZOASBZ463H550,0001394395,Deutsche Bank AG,7LTWFZYICNSX8D621K86
8,Emerging Markets Local Income Portfolio,TJHVIYYZOASBZ463H550,0001394395,JPMorgan Chase Bank N.A.,7H6GLXDRUGQFU57RNE97
9,Emerging Markets Local Income Portfolio,TJHVIYYZOASBZ463H550,0001394395,"Citigroup Global Markets, Inc.",MBNUM2BPBDO7JBLYG310


In [ ]:
def get_all_counterparties():
  all_counterparties = None
  has_more_filings = True
  filing_counter = 0
  search_from = 0
  search_size = 10

  while has_more_filings:
    search_query = {
      "query": "invstOrSecs.derivativeInfo.swapDeriv.derivCat:SWP AND filedAt:[2020-01-01 TO 2020-12-31]",
      # offset or starting point for search results
      "from": search_from,
      # number of filings returned per search request
      "size": search_size,
      # sort result by the filedAt field, start with the most recently filed filing
      "sort": [{ "filedAt": { "order": "desc" } }]
    }

    response = nportApi.get_data(search_query)

    if len(response['filings']) == 0:
      break

    for filing in response['filings']:
      result = extract_counterparties(filing)
      if all_counterparties is None:
        all_counterparties = result
      else:
        all_counterparties = pd.concat([all_counterparties, result], ignore_index=True)

    filing_counter += len(response['filings'])

    if filing_counter % 100 == 0:
      print('{} filings downloaded'.format(filing_counter))

    search_from += search_size

  all_counterparties.drop_duplicates(subset=['fundLei', 'counterparytName', 'counterpartyLei'], inplace=True)

  all_counterparties.reset_index(drop=True, inplace=True)

  print('Done. {} filings downloaded'.format(filing_counter))

  return all_counterparties

In [ ]:
all_counterparties = get_all_counterparties()

100 filings downloaded
200 filings downloaded
300 filings downloaded
Done. 387 filings downloaded


In [ ]:
all_counterparties

,fundName,fundLei,fundCik,counterparytName,counterpartyLei
0,AIM Investment Funds (Invesco Investment Funds),Y5W0BJB7U2X9V6NIC803,0000826644,LCH.Clearnet LLC,WAM6YERMS7OXFZUOY219
1,AIM Investment Funds (Invesco Investment Funds),Y5W0BJB7U2X9V6NIC803,0000826644,Goldman Sachs International,W22LROWP2IHZNBB6K528
2,AIM Investment Funds (Invesco Investment Funds),Y5W0BJB7U2X9V6NIC803,0000826644,CME Group Inc,LCZ7XYGSLJUHFXXNXD88
3,AIM Investment Funds (Invesco Investment Funds),Y5W0BJB7U2X9V6NIC803,0000826644,Chicago Mercantile Exchange Inc,SNZ2OJLFK8MNNCLQOF39
4,AIM Investment Funds (Invesco Investment Funds),Y5W0BJB7U2X9V6NIC803,0000826644,Intercontinental Exchange Inc,5493000F4ZO33MV32P92
...,...,...,...,...,...
809,DFA INVESTMENT DIMENSIONS GROUP INC,549300XGRTBJCE41BD26,0000355437,Citigroup Global Markets Inc.,MBNUM2BPBDO7JBLYG310
810,DFA INVESTMENT DIMENSIONS GROUP INC,549300XGRTBJCE41BD26,0000355437,Deutsche Bank Aktiengesellschaft,7LTWFZYICNSX8D621K86
811,DFA INVESTMENT DIMENSIONS GROUP INC,549300XGRTBJCE41BD26,0000355437,"Merrill Lynch, Pierce, Fenner & Smith Incorporated",8NAV47T0Y26Q87Y0QP81
812,NEUBERGER BERMAN INCOME FUNDS,54930087F7MMJ21RIJ90,0000723620,"JPMorgan Chase Bank, National Association",7H6GLXDRUGQFU57RNE97


From the 814 combinations of fund names and their counterparties extracted from 2020 N-PORT filings, we complete our tutorial with some examples showing how to filter and analyze the dataset.

Let's say we want to find all funds that use UBS Switzerland (LEI: 549300WOIFUSNYH0FL22) as counterparty to their swap derivatives.

In [ ]:
print('Funds that use UBS Switzerland as counterparty to their swaps:')
all_counterparties[(all_counterparties['counterpartyLei']=='549300WOIFUSNYH0FL22')]

Funds that use UBS Switzerland as counterparty to their swaps:


,fundName,fundLei,fundCik,counterparytName,counterpartyLei
616,ABERDEEN GLOBAL INCOME FUND INC,549300V7ZDFLSK6WD203,0000876717,UBS Switzerland AG,549300WOIFUSNYH0FL22
621,ABERDEEN ASIA-PACIFIC INCOME FUND INC,549300J666ZH67203572,0000790500,UBS Switzerland AG,549300WOIFUSNYH0FL22
740,Global Macro Absolute Return Advantage Portfolio,NKY7JRBKJHQQ68KJ6252,0001493214,UBS Switzerland AG,549300WOIFUSNYH0FL22
752,Global Macro Capital Opportunities Portfolio,549300SKGAQ18F9L3O15,0001588812,UBS Switzerland AG,549300WOIFUSNYH0FL22
767,Global Macro Portfolio,XY6HWOQF1NBIQHYB7D92,0000918706,UBS Switzerland AG,549300WOIFUSNYH0FL22


If we want to find all funds that use any UBS entity as counterparty, not only UBS Switzerland, we can broaden our search using the `.str.contains()` function to look for any appearance of the term `UBS` in the `counterpartyName` cell.

In [ ]:
print('Funds that use an UBS entity as counterparty to their swaps:')
all_counterparties[all_counterparties['counterparytName'].str.contains('UBS')]

Funds that use an UBS entity as counterparty to their swaps:


,fundName,fundLei,fundCik,counterparytName,counterpartyLei
28,AIM Investment Funds (Invesco Investment Funds),Y5W0BJB7U2X9V6NIC803,0000826644,UBS AG,BFM8T61CT2L1QCEMIK50
34,AIM Investment Funds (Invesco Investment Funds),Y5W0BJB7U2X9V6NIC803,0000826644,UBS Securities LLC,T6FIZBDPKLYJKFCRVK44
109,ProFunds,549300HM4OOUYSE5DP77,0001039803,UBS SECURITIES LLC,T6FIZBDPKLYJKFCRVK44
114,FINANCIAL INVESTORS TRUST,549300XD6LCPI5YWPZ35,0000915802,UBS Group AG,N/A
149,"T. ROWE PRICE MULTI-STRATEGY TOTAL RETURN FUND, INC.",5493003IUKLD53D0ET20,0001707770,UBS SECURITIES LLC,T6FIZBDPKLYJKFCRVK44
162,BlackRock Funds,549300OZUEVJZHOBFP42,0000844779,UBS AG,BFM8T61CT2L1QCEMIK50
210,Direxion Shares ETF Trust,549300M501IVJM50FG12,0001424958,UBS AG,BFM8T61CT2L1QCEMIK50
299,Credit Suisse Commodity Strategy Funds,549300KFU6FOOSD82072,0001291446,UBS,N/A
432,John Hancock Investment Trust,549300WRGYBW55IKGU23,0000022370,UBS SECURITIES LLC,N/A
616,ABERDEEN GLOBAL INCOME FUND INC,549300V7ZDFLSK6WD203,0000876717,UBS Switzerland AG,549300WOIFUSNYH0FL22


Now let's compare this to the number of funds that use Goldman Sachs as their counterparty.

In [ ]:
print('Funds that use Goldman Sachs as counterparty to their swaps:')
all_counterparties[(all_counterparties['counterpartyLei']=='FOR8UP27PHTHYVLBNG30') | (all_counterparties['counterparytName'].str.contains('Goldman'))]

Funds that use Goldman Sachs as counterparty to their swaps:


,fundName,fundLei,fundCik,counterparytName,counterpartyLei
1,AIM Investment Funds (Invesco Investment Funds),Y5W0BJB7U2X9V6NIC803,0000826644,Goldman Sachs International,W22LROWP2IHZNBB6K528
18,AIM International Mutual Funds (Invesco International Mutual Funds),549300CGSTEJJ7H1ET84,0000880859,Goldman Sachs International,W22LROWP2IHZNBB6K528
20,Invesco Securities Trust,549300D73HZMJHRYEH02,0001560704,Goldman Sachs Bank USA,KD3XUN7C6T14HNAYLU02
25,AIM Investment Funds (Invesco Investment Funds),Y5W0BJB7U2X9V6NIC803,0000826644,Goldman Sachs Bank USA,KD3XUN7C6T14HNAYLU02
64,PACE SELECT ADVISORS TRUST,549300TPB2UX3QR3SC50,0000930007,Goldman Sachs & Co. LLC,FOR8UP27PHTHYVLBNG30
65,PACE SELECT ADVISORS TRUST,549300TPB2UX3QR3SC50,0000930007,Goldman Sachs International,W22LROWP2IHZNBB6K528
85,Hartford Funds Exchange-Traded Trust,549300IA3XO4BB2LUL19,0001501825,Goldman Sachs International,W22LROWP2IHZNBB6K528
96,HARTFORD MUTUAL FUNDS INC/CT,549300FJ4Q3QXIS6ZN84,0001006415,Goldman Sachs International,W22LROWP2IHZNBB6K528
103,HARTFORD MUTUAL FUNDS INC/CT,549300FJ4Q3QXIS6ZN84,0001006415,Goldman Sachs & Co. LLC,FOR8UP27PHTHYVLBNG30
142,"T. ROWE PRICE MULTI-STRATEGY TOTAL RETURN FUND, INC.",5493003IUKLD53D0ET20,0001707770,GOLDMAN SACHS & CO. LLC,FOR8UP27PHTHYVLBNG30


## Extracting the Address & Phone Number of Funds

If you are looking to extract the fund's address, phone number and series information such as series name, LEI and series ID, the following code will help you.

Such information can be found inside the `genInfo` element of the N-PORT XML file. The same information is located in the `genInfo` object of the JSON-converted data, as illustrated in the comparison below.

| XML Example | XML Converted to JSON |
| -- | -- |
| ![nport-fund-general-info](https://i.imgur.com/r4iU4YD.png) | ![general-info-converted-to-json](https://i.imgur.com/q0xYsLN.png) |

In [ ]:
print("Address, phone number and other general fund information:")
pprint(sample_filing["genInfo"])

Address, phone number and other general fund information:
{
  "regName": "Emerging Markets Local Income Portfolio",
  "regFileNumber": "811-22048",
  "regCik": "0001394395",
  "regLei": "TJHVIYYZOASBZ463H550",
  "regStreet1": "TWO INTERNATIONAL PLACE",
  "regCity": "BOSTON",
  "regStateConditional": {
    "regCountry": "US",
    "regState": "US-MA"
  },
  "regZipOrPostalCode": "02110",
  "regPhone": "617-482-8260",
  "seriesName": "Emerging Markets Local Income Portfolio",
  "seriesId": "S000017994",
  "seriesLei": "TJHVIYYZOASBZ463H550",
  "repPdEnd": "2023-10-31",
  "repPdDate": "2023-01-31",
  "isFinalFiling": "N"
}
